It is 7pm on a Saturday during February half term, the busiest night of the busiest week of the bowling year. Every lane in a 24-lane centre is booked back-to-back. The machines have been running since 10am with no recovery time between sessions, and many have not had a full service since the Christmas rush.

Your task is to build a model that predicts which pinsetters are likely to fail in the next 24 hours, so a technician can be called before a lane goes dark.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('laneguard_sensor_logs.csv')

print(f"Rows: {df.shape[0]}   Columns: {df.shape[1]}")
print(f"Failure rate: {df['failure_within_24h'].mean():.1%}")
print()
print("Missing values per column:")
print(df.isnull().sum().to_string())

One column has missing values: `pinsetter_drive_temp`.

**Predict:** Do you think the rows with a missing temperature reading are more likely, less likely, or about as likely to have a failure as the rows where temperature was recorded? Write your reasoning in the cell below before running the next cell.

*Write your answer here.*

In [ ]:
# Compare failure rates between rows with and without a temperature reading
null_rows     = df[df['pinsetter_drive_temp'].isnull()]
non_null_rows = df[df['pinsetter_drive_temp'].notna()]

rate_null    = null_rows['failure_within_24h'].mean()
rate_present = non_null_rows['failure_within_24h'].mean()

# How many of the total failures sit in the null rows?
pct_of_failures = null_rows['failure_within_24h'].sum() / df['failure_within_24h'].sum()

print(f"Failure rate, temp missing:  {rate_null:.1%}")
print(f"Failure rate, temp recorded: {rate_present:.1%}")
print(f"Ratio: {rate_null / rate_present:.1f}x")
print()
print(f"Rows with missing temp contain {pct_of_failures:.0%} of all failures in the dataset.")

The rows with a missing temperature reading fail at more than four times the rate of rows with a recorded one.

**Predict:** Given that result, what do you think is causing the temperature values to go missing? Write your reasoning in the cell below before continuing.

*Write your answer here.*

Once you have written your prediction, here are the facts that explain the result:

**1.** The temperature sensor trips its self-protection circuit when the motor surface exceeds approximately 92 degrees Celsius. That is the condition that precedes failure. The sensor goes silent precisely when it has the most to say.

**2.** The column mean is approximately 66 degrees, a normal operating temperature. The rows where the sensor tripped were running more than 25 degrees above that.

**3.** Those rows with missing temperature contain approximately a third of all the failures in the dataset, in only 10% of the rows. They are not randomly absent. They are the most failure-heavy rows in the data.

Hold those three facts. Each of the three cleaning strategies handles those rows differently, and one result is going to be counterintuitive.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, recall_score

TARGET = 'failure_within_24h'
BASE   = ['pinsetter_drive_temp', 'pinsetter_chassis_vibration',
          'days_since_service', 'cycles_since_service']

def fit_and_score(X_tr, y_tr, X_te, y_te):
    # Up-weight failure cases: 9% failure rate means the model will ignore
    # them without this correction
    w = np.where(y_tr == 1, (y_tr == 0).sum() / (y_tr == 1).sum(), 1.0)
    # n_estimators=300 gives stable results: the qualitative ordering
    # (drop worst, mean+flag best F1) only holds reliably at this setting
    m = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05,
                                   max_depth=4, subsample=0.8, random_state=42)
    m.fit(X_tr, y_tr, sample_weight=w)
    p = m.predict(X_te)
    return m, f1_score(y_te, p, zero_division=0), \
              recall_score(y_te, p, zero_division=0), p

# Split before cleaning: all three strategies must face the same test rows
idx_tr, idx_te = train_test_split(
    df.index, test_size=0.2, random_state=42, stratify=df[TARGET])
train, test = df.loc[idx_tr].copy(), df.loc[idx_te].copy()

# Fill value from training data only, never from test rows
fill = train['pinsetter_drive_temp'].mean()
print(f"Fill value: {fill:.1f} degrees C  |  Test failures: {test[TARGET].sum()}")

In [ ]:
# Drop training rows where temperature was not recorded
train_drop = train.dropna(subset=['pinsetter_drive_temp'])

# Test set: fill its few nulls with the training mean regardless of strategy
# so all three models are evaluated on identical rows
test_drop  = test.copy()
test_drop['pinsetter_drive_temp'] = test_drop['pinsetter_drive_temp'].fillna(fill)

m_drop, f1_drop, rec_drop, p_drop = fit_and_score(
    train_drop[BASE], train_drop[TARGET],
    test_drop[BASE],  test_drop[TARGET])

total       = int(test_drop[TARGET].sum())
caught_drop = int((p_drop[test_drop[TARGET] == 1] == 1).sum())
print(f"Drop:  F1={f1_drop:.3f}  Recall={rec_drop:.1%}  "
      f"Caught {caught_drop} of {total} failures")

9 failures caught out of 55. That means 46 failures were not predicted. Those are 46 lanes that will break tonight with no technician called.

Recall fact 3 from above: the null rows contain approximately a third of all the failures in the dataset.

**Interpret:** The drop strategy removed those rows from training. Using that fact, write one sentence in the cell below explaining why this strategy produced the worst recall of the three.

*Write your answer here.*

In [ ]:
# Replace each missing temperature with 66 degrees, the training mean
train_mean = train.copy()
train_mean['pinsetter_drive_temp'] = train_mean['pinsetter_drive_temp'].fillna(fill)
test_mean  = test.copy()
test_mean['pinsetter_drive_temp']  = test_mean['pinsetter_drive_temp'].fillna(fill)

m_mean, f1_mean, rec_mean, p_mean = fit_and_score(
    train_mean[BASE], train_mean[TARGET],
    test_mean[BASE],  test_mean[TARGET])

caught_mean = int((p_mean[test_mean[TARGET] == 1] == 1).sum())
print(f"Mean:  F1={f1_mean:.3f}  Recall={rec_mean:.1%}  "
      f"Caught {caught_mean} of {total} failures")

17 failures caught, up from 9. Mean imputation performed better than drop, even though it replaced every temperature reading above 92 degrees with 66 degrees, a value that looks like a normally running motor. That result is probably not what you predicted.

**Predict:** The third strategy adds a column called `temp_was_missing` alongside the filled temperature. Given what you now know about those rows, do you expect this to improve recall, worsen it, or leave it the same? Write your reasoning in the cell below before running the next cell.

*Write your answer here.*

In [ ]:
# Record which rows had a null before overwriting the temperature
train_flag = train.copy()
train_flag['temp_was_missing'] = train_flag['pinsetter_drive_temp'].isnull().astype(int)
train_flag['pinsetter_drive_temp'] = train_flag['pinsetter_drive_temp'].fillna(fill)

test_flag = test.copy()
test_flag['temp_was_missing'] = test_flag['pinsetter_drive_temp'].isnull().astype(int)
test_flag['pinsetter_drive_temp'] = test_flag['pinsetter_drive_temp'].fillna(fill)

FEATURES_FLAG = BASE + ['temp_was_missing']
m_flag, f1_flag, rec_flag, p_flag = fit_and_score(
    train_flag[FEATURES_FLAG], train_flag[TARGET],
    test_flag[FEATURES_FLAG],  test_flag[TARGET])

caught_flag = int((p_flag[test_flag[TARGET] == 1] == 1).sum())
print(f"Mean+flag:  F1={f1_flag:.3f}  Recall={rec_flag:.1%}  "
      f"Caught {caught_flag} of {total} failures")

Mean and mean+flag caught the same number of failures: 17 out of 55. The recall is identical. But the F1 is slightly higher for mean+flag, which means it made fewer false alarms while catching the same failures.

The same recall does not mean the two models learned the same thing. Run the SHAP plots below and look for what is different between them.

In [ ]:
import shap
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

for label, model, X_te in [
    ('mean imputation',  m_mean, test_mean[BASE]),
    ('mean + indicator', m_flag, test_flag[FEATURES_FLAG])]:

    explainer = shap.TreeExplainer(model)
    sv        = explainer.shap_values(X_te)

    plt.figure(figsize=(7, 3))
    shap.summary_plot(sv, X_te, plot_type='bar', show=False)
    plt.title(f'Feature importance: {label}', fontsize=11)
    plt.tight_layout()
    plt.show()

In the mean model, SHAP says temperature is the most important feature. That is technically correct: the model used that column heavily. But roughly 300 of those temperature values were filled in by the cleaning script, not recorded by the sensor. The model does not know which ones. SHAP cannot tell you either.

In the mean+flag model, `temp_was_missing` ranks third. That column contains no temperature data at all, only a 0 or a 1 recording whether a reading was absent. It is one of the three most useful predictors of failure. The model has learned that the sensor going silent is itself an event worth paying attention to.

**Reflect:** If you showed the mean imputation model's SHAP explanation to the centre's operations manager, they would see "temperature is the most important factor." That statement is accurate. Write in the cell below: in what sense is it also misleading, and what would a more honest explanation say?

*Write your answer here.*

Both models were built after you already knew the outcome. In a production pipeline you want to catch the non-random null before imputation runs, not after you have trained three models and compared the results. Pydantic is a Python library for data validation: you define what a valid data record looks like as a typed class, and it checks every incoming row against that definition before anything downstream in the pipeline touches the data. The class below adds a custom validator for the temperature column that flags the kind of null you saw in this dataset.

One thing to know before reading the code: pandas represents missing numeric values as `float('nan')`, not as Python `None`. The `model_validator` at the top of the class converts any `nan` values to `None` before the temperature check runs, so the validator works correctly on real dataframe rows.

In [ ]:
from pydantic import BaseModel, field_validator, model_validator, ValidationError
from typing import Optional

class LaneReading(BaseModel):
    lane_id:                     int
    pinsetter_drive_temp:        Optional[float] = None  # permitted to be absent
    pinsetter_chassis_vibration: float
    days_since_service:          int
    cycles_since_service:        int

    @model_validator(mode='before')
    @classmethod
    def convert_nan_to_none(cls, values):
        # Pandas passes NaN for missing floats, not None: normalise before field validation runs
        return {k: None if isinstance(v, float) and np.isnan(v) else v
                for k, v in values.items()}

    @field_validator('pinsetter_drive_temp')
    @classmethod  # required by Pydantic v2
    def flag_non_random_null(cls, v):
        if v is None:
            # Stop here: this null is failure-correlated, not random noise
            raise ValueError(
                "Null temperature: sensor protection circuit may have tripped. "
                "Add temp_was_missing indicator column before imputing.")
        return v

Look at the `flag_non_random_null` method in the class above. Before running the tests, write your prediction in the cell below: what do you expect to happen when a row with a missing temperature is passed to `LaneReading`? What will the output say, and at what point in the pipeline will it happen?

*Write your answer here.*

In [ ]:
# A complete row should pass without any error
healthy = df.dropna().iloc[0].to_dict()
r = LaneReading(**healthy)
print(f"Valid: lane {r.lane_id} at {r.pinsetter_drive_temp:.1f} degrees C")

# A row with a missing temperature should be stopped before imputation runs
null_row = df[df['pinsetter_drive_temp'].isnull()].iloc[0].to_dict()
try:
    LaneReading(**null_row)
except ValidationError as e:
    print(f"\nStopped before imputation:")
    print(f"  {e.errors()[0]['msg']}")

The validator currently stops the pipeline completely when a null temperature arrives. That is useful during development, but in production you may want to let the row through while still flagging it, passing it to the mean+flag path rather than rejecting it outright.

**Challenge:** Write a new class called `LaneReadingV2` that accepts a null temperature without raising an error. Instead of stopping the pipeline, the class should automatically set a boolean field called `temp_was_missing` to `True` when temperature is null, and `False` when it is present. Every row should pass validation. The indicator should be set correctly without any code outside the class.

*Hint: use `model_validator(mode='before')`, the same pattern as `convert_nan_to_none` above, to convert NaN to None and set the `temp_was_missing` flag in one step. You do not need the error-raising validator from the original class.*

In [ ]:
# Write your LaneReadingV2 class below


# Uncomment to test your solution once written:
# null_row = df[df['pinsetter_drive_temp'].isnull()].iloc[0].to_dict()
# r = LaneReadingV2(**null_row)
# print(f"temp_was_missing={r.temp_was_missing}")  # should be True
# healthy = df.dropna().iloc[0].to_dict()
# r2 = LaneReadingV2(**healthy)
# print(f"temp_was_missing={r2.temp_was_missing}")  # should be False


Three strategies, three different things learned.

Dropping the rows removed the evidence. The model was trained without the cases that mattered most and learned a quieter, more optimistic version of reality: one where failures are less common and less connected to temperature than they actually are.

Mean imputation kept the rows but corrupted the signal. The model performed better because it had more failure cases to train on, but its SHAP explanation told a story about temperature that was only partly true. Roughly 300 of those temperature readings were estimates, not measurements, and the explanation had no way to say so.

Mean+flag kept the rows, corrupted the temperature in the same way, but told the model that the corruption happened. The sensor going silent became a feature in its own right. The model learned from the absence of data, not just its presence.

The cleaning decision changed what the model was allowed to learn. SHAP reported what it learned. Pydantic is how you catch the problem before training runs rather than after, so the decision about which strategy to use is made deliberately, in the pipeline, with a record of why.